In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

N_PEERS = 300
N_ITERS = 20
T_END = 300
TARGET_DISCOVERY_RATE = 99

T_OFFSET = 10.8

In [ ]:
def read_reach_data(target: str) -> list[list[tuple[float, float]]]:
    """
    Read reachability data from experiment files.
    Returns: list of measurements for each iteration, where each measurement is (time, reachability%)
    """
    data = []
    for i in range(N_ITERS):
        with open(f'../scale/{target}/{N_PEERS}/r_{i}.txt', 'r') as f:
            time_init = 0
            measurements = []
            
            for line in f:
                parts = line.strip().split()
                if parts[1] == 'N/A':
                    continue

                time = int(parts[0])
                if time_init == 0:
                    time_init = time
                execution_time = (time - time_init) / 1000
                reachability = float(parts[1])
                
                measurements.append((execution_time, reachability * 100))
            data.append(measurements)
    return data

In [ ]:
def time_split_discovery_time(data: list[tuple[float, float]]) -> list[float]:
    """
    Split data into 10-second windows and calculate discovery time for each window.
    Returns: list of discovery times (time to reach TARGET_DISCOVERY_RATE) for each window
    """
    result = []
    time_now = 0.1
    split_discovery_time = 0.0
    
    for time, reach in data:
        if time >= time_now + 10:
            time_now += 10
            if split_discovery_time == 0.0:
                result.append(10.0)  # Never reached target in this window
            else:
                result.append(split_discovery_time)
                
            split_discovery_time = 0.0
            continue

        if reach > TARGET_DISCOVERY_RATE and split_discovery_time == 0.0:
            split_discovery_time = time - time_now + 0.1
    return result

In [ ]:
def parse_reach_all_seeds(target: str) -> list[list[float]]:
    """
    Parse reachability data across all seeds and organize by time window.
    Returns: list where each element contains discovery times from all seeds for that window
    """
    dt_flow_seeds = [time_split_discovery_time(data) for data in read_reach_data(target)]
    min_length = min(len(n) for n in dt_flow_seeds)
    
    result = []
    for i in range(min_length):
        result.append([n[i] for n in dt_flow_seeds])
    return result

In [ ]:
def read_ifstat(target: str) -> list[list[tuple[list[float], list[float]]]]:
    """
    Read network interface statistics from log files.
    Returns: list of seeds, each containing list of (time_points, netusages) tuples for each peer
    """
    result = []
    for seed in range(N_ITERS):
        file_paths = [f'../scale/{target}/{N_PEERS}/{seed}/ifstat_h{i+1}.log' for i in range(N_PEERS)]

        data = []
        for file_path in file_paths:
            time_points = []
            netusages = []
            with open(file_path, 'r') as f:
                # Skip first two header lines
                next(f, None)
                next(f, None)

                for line_no, line in enumerate(f):
                    execution_time = line_no / 10 - T_OFFSET
                    netusage = sum(float(v) for v in line.strip().split())
                    
                    # Skip data before experiment burst
                    if execution_time < 0.0:
                        continue
                    
                    time_points.append(execution_time)
                    netusages.append(netusage)
            data.append((time_points, netusages))

        result.append(data)
    return result

In [ ]:
def time_split_sum_netuse(time_points: list[float], netusages: list[float]) -> list[float]:
    """
    Split network usage into 10-second windows and sum the usage in each window.
    Returns: list of total network usage (in kB) for each 10-second window
    """
    result = []
    time_now = 0.0
    split_sum = 0.0
    
    for t, n in zip(time_points, netusages):
        while t >= time_now + 10:
            time_now += 10
            result.append(split_sum)
            split_sum = 0

        split_sum += n / 8 * 0.1 / 1000  # Convert to MB
    return result

In [ ]:
def time_split_netuse_per_peer(data: list[tuple[list[float], list[float]]]) -> list[float]:
    """
    Calculate average network usage per peer for each time window.
    Returns: list of average network usage per peer (in kB) for each window
    """
    data_hosts = [time_split_sum_netuse(t, n) for t, n in data]
    min_length = min(len(h) for h in data_hosts)
    data_hosts_trimmed = [l[:min_length] for l in data_hosts]

    result = []
    for i in range(min_length):
        total_usage = sum(v[i] for v in data_hosts_trimmed)
        avg_per_peer = total_usage / (10 * (i + 1))
        result.append(avg_per_peer)
    return result

In [ ]:
def parse_netuse_all_seeds(target: str) -> list[list[float]]:
    """
    Parse network usage data across all seeds and organize by time window.
    Returns: list where each element contains network usage from all seeds for that window
    """
    netuse_flow_seeds = [time_split_netuse_per_peer(data) for data in read_ifstat(target)]
    min_length = min(len(n) for n in netuse_flow_seeds)

    result = []
    for i in range(min_length):
        result.append([n[i] for n in netuse_flow_seeds])
    return result

In [ ]:

COLORS = ["#00265C", "#9C1B2D", "#C7A705"]

def draw_stacked_boxplots_only(
    data_top: list[list[list[float]]],
    data_bottom: list[list[list[float]]],
    x_labels: list[str],
    y_label_top: str,
    y_label_bottom: str,
    group_labels: list[str],
    x_label: str,
):
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 6,
        "axes.titlesize": 6,
        "axes.labelsize": 6,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "legend.fontsize": 6,
        "figure.titlesize": 6,
    })

    n_groups = len(data_top)
    n_positions = len(x_labels)
    
    # Calculate box width and spacing
    box_width = 0.8 / n_groups
    group_offset = np.arange(n_groups) - (n_groups - 1) / 2
    group_offset = group_offset * box_width
    
    fig, (ax_top, ax_bot) = plt.subplots(2, 1, sharex=True, figsize=(8, 4))

    # --- Top plot: Boxplots ---
    for i, (group, label) in enumerate(zip(data_top, group_labels)):
        positions = np.arange(n_positions) + group_offset[i]
        bp = ax_top.boxplot(
            group,
            positions=positions,
            widths=box_width * 0.9,
            patch_artist=True,
            flierprops=dict(
                marker='.',
                markerfacecolor=COLORS[i],
                markeredgecolor=COLORS[i],
                markersize=3,
                linestyle='none'
            ),
            medianprops=dict(color=COLORS[i], linewidth=1),
            boxprops=dict(facecolor=COLORS[i], edgecolor=COLORS[i], alpha=0.7),
            whiskerprops=dict(color=COLORS[i]),
            capprops=dict(color=COLORS[i]),
        )
        # Add legend entry
        ax_top.plot([], [], color=COLORS[i], linewidth=2, alpha=0.7, label=label)

    def custom_formatter(x_val, pos):
        val_str = '{:g}'.format(x_val)
        if 0 < abs(x_val) < 1:
            # Removes the '0' before the decimal point
            return val_str.replace("0", "", 1)
        elif -1 < abs(x_val) < 0:
            # Removes the '0' between '-' and '.'
            return val_str.replace("0", "", 1)
        else:
            return val_str

    # Apply the formatter to the y-axis
    ax_top.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))

    ax_top.set_ylabel(y_label_top, labelpad=1)
    ax_top.tick_params(bottom=False)
    ax_top.grid(axis='both', linestyle='--', alpha=0.4)
    ax_top.legend(loc='upper left', frameon=False)

    # --- Bottom plot: Boxplots ---
    for i, (group, label) in enumerate(zip(data_bottom, group_labels)):
        positions = np.arange(n_positions) + group_offset[i]
        bp = ax_bot.boxplot(
            group,
            positions=positions,
            widths=box_width * 0.9,
            patch_artist=True,
            flierprops=dict(
                marker='.',
                markerfacecolor=COLORS[i],
                markeredgecolor=COLORS[i],
                markersize=3,
                linestyle='none'
            ),
            medianprops=dict(color=COLORS[i], linewidth=1),
            boxprops=dict(facecolor=COLORS[i], edgecolor=COLORS[i], alpha=0.7),
            whiskerprops=dict(color=COLORS[i]),
            capprops=dict(color=COLORS[i]),
        )

    x_numeric = np.arange(n_positions)
    ax_bot.set_ylabel(y_label_bottom, labelpad=1)
    ax_bot.set_xlabel(x_label)
    ax_bot.set_xticks(x_numeric)
    ax_bot.set_xticklabels(x_labels)
    ax_bot.tick_params(axis='x', labelrotation=45)
    ax_bot.grid(axis='both', linestyle='--', alpha=0.4)
    ax_bot.set_ylim(0, 9)

    plt.tight_layout()
    plt.savefig('scale_trend_b.jpg', dpi=600, bbox_inches='tight', pad_inches=0)

In [ ]:
targets = ['client-server', 'dev-eval-trickle', 'dev-v2']

top    = [parse_netuse_all_seeds(target) for target in targets]
bottom = [parse_reach_all_seeds(target) for target in targets]
min_length = min(min([len(v) for v in top]), min([len(v) for v in bottom]))
labels = [f"{10*(i+1)}" for i in range(min_length)]

draw_stacked_boxplots_only(
    [v[:min_length] for v in top], 
    [v[:min_length] for v in bottom],
    x_labels=labels,
    y_label_top="Network (MB)",
    y_label_bottom=f"Discovery Time (s)",
    group_labels=['Client-Server', 'Trickle', 'Ours'],
    x_label="Number of Peers",
)